# 05 -- AlphaZero (Step 9)

Pure self-play from a randomly initialized network: no SFT data, no Stockfish, game results only. Each
**round** plays `games_per_round` games at once, every move chosen by MCTS with the current network
(`search/mcts.py`); every searched position goes into a replay buffer with two targets -- the search's visit
shares (policy) and the game's final score for the side to move (value) -- and the network takes K
minibatch steps toward them (`train/az_train.py`). Search makes the network's play stronger, the network
learns to predict that stronger play, and the next round's search starts from a better network.

Then kept checkpoints are rated into `results/results.csv` like every other arm: up to 4 per cap stage,
each stage's last included. The AlphaZero run is its own series on the Elo-vs-compute chart, starting at 0
GPU-hours.

Settings: `configs/alphazero.yaml` (all **provisional** until the smoke run is reviewed, plan Sec 5 item 4),
`configs/base.yaml`, `configs/eval.yaml`.

**Cap, staged 9 / 18 / 30 GPU-hours.** `max_gpu_hours` in `configs/alphazero.yaml` is not part of the run's
saved settings: raise it, push, pull, and rerun this notebook to resume the same run into the next stage.

**First run `SMOKE_TEST = True`** (2 tiny rounds in a separate `_smoke` folder, plus a timing probe at the
real batch size; no eval). Report its timings before the real run: they decide the round settings.

**Safe to interrupt.** Every round is saved; rerunning resumes the run. Evals skip checkpoints already rated.

Headless on the pod (in tmux; `docs/runpod_runbook.md`):
`jupyter nbconvert --to notebook --execute notebooks/05_train_alphazero.ipynb --output-dir /workspace/chess-ai/logs --ExecutePreprocessor.timeout=-1`

**Needs a GPU** (L4 pod). Never `git pull` on a pod that is running a job.

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
# Before each session, in a pod terminal (code is never edited on the pod):
#   cd /workspace/Chess_Bot && git fetch && git checkout BRANCH && git reset --hard origin/BRANCH
import os
import subprocess

BRANCH = "step-9-alphazero"
REPO_DIR = "/workspace/Chess_Bot"
%cd {REPO_DIR}
!git log -1 --oneline
on = subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True).stdout.strip()
if on != BRANCH:
    print(f"WARNING: the repo is on {on!r}, not {BRANCH!r} -- check out the right branch before a real run")
!pip install -q -r requirements.txt

import yaml
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
import torch
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())   # eval opponents only; training never uses it
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
CPU_MODEL = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "unknown")
print("GPU:", GPU, "| CPU:", CPU_MODEL, "| cores:", os.cpu_count())

## Parameters
Everything comes from `configs/alphazero.yaml` except the flags below. The run folder is named after the
network size, seed and `tag`; rerunning with the same settings resumes it, and different settings in the
same folder are refused (change `tag` when you change a setting).

In [ ]:
import dataclasses
import json

from train.az_train import eval_rounds, list_checkpoints, load_configs, run_name, train

RUN_TRAINING = True   # False = skip straight to the curves and the evaluation
SMOKE_TEST = False    # True = 2 tiny rounds in a separate "_smoke" folder + a timing probe; no eval
MAX_ROUNDS = None     # rounds to play this session (None = until the cap); the run resumes later
EVAL_ROUNDS = None    # e.g. [5, 10, 20]; None = up to 4 kept checkpoints per cap stage (eval_rounds)
OPPONENTS = ["random", "greedy", "step_20000"]   # move up (sf_1320, sf_1500, sf_1700) as it gets stronger

CFG, NET, MCTS, MAX_GPU_HOURS = load_configs("configs/alphazero.yaml")
FULL_CFG, FULL_MCTS = CFG, MCTS              # the real settings, for the smoke run's timing projection
if SMOKE_TEST:
    CFG = dataclasses.replace(CFG, rounds=2, games_per_round=8, max_plies=40, buffer_positions=5_000,
                              checkpoint_every=1)
    MCTS = dataclasses.replace(MCTS, simulations=8)
    MAX_GPU_HOURS = None
RUN = run_name(CFG, NET) + ("_smoke" if SMOKE_TEST else "")
RUN_DIR = os.path.join(ROOT, "checkpoints", "alphazero", RUN)
print("net:", NET)
print("search:", MCTS)
print("train:", CFG)
print("run folder:", RUN_DIR, "| GPU-hour cap:", MAX_GPU_HOURS)

## Train
One line per round: self-play W/D/L and how games ended, policy loss, value loss, policy entropy (how
spread out the network's move choices are -- a crash toward 0 means collapse), and where the time went.
Stops at the GPU-hour cap; that round is kept for evaluation.

In [ ]:
if RUN_TRAINING:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    summary = train(CFG, NET, MCTS, RUN_DIR, max_gpu_hours=MAX_GPU_HOURS, max_rounds=MAX_ROUNDS,
                    extra_info={"gpu": GPU, "cpu": CPU_MODEL, "cores": os.cpu_count()})
    print(json.dumps(summary, indent=2))
else:
    print("RUN_TRAINING is False -- skipping training")

## Smoke run: timing at the real settings (smoke test only)
The tiny rounds above check the loop; their timing doesn't scale to real rounds (few games per network
call, few simulations). This probe times ONE move of `games_per_round` games at the real simulation
count, with an untrained network of the real size -- the unit a real round is made of -- and one
update step at the real batch size. From those it projects a real round. Report these numbers before
the first real run: together with the provisional defaults they set how many rounds the 9-GPU-h stage buys.

In [ ]:
if SMOKE_TEST:
    import time
    import chess
    import pandas as pd
    import numpy as np
    import random
    from eval.match import random_opening
    from model.az_net import AZEvaluator, AZNet
    from search.mcts import play_games, run_search
    from train.az_train import ReplayBuffer, game_examples, update, update_steps

    log = pd.read_csv(os.path.join(RUN_DIR, "az_log.csv"))
    print(log[["round", "positions", "avg_plies", "selfplay_seconds", "update_seconds"]].to_string(index=False))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = AZNet(NET).to(device).eval()
    ev = AZEvaluator(net, device)
    rng = random.Random(0)
    boards = []
    for _ in range(FULL_CFG.games_per_round):
        b = chess.Board()
        for mv in random_opening(rng, 10):
            b.push_uci(mv)
        boards.append(b)
    run_search(boards[:4], ev, dataclasses.replace(FULL_MCTS, simulations=2))   # warm-up (CUDA init)
    t = time.perf_counter()
    run_search(boards, ev, FULL_MCTS, add_noise=True, rng=np.random.default_rng(0))
    sec_per_move = time.perf_counter() - t

    games = play_games(ev, 4, dataclasses.replace(FULL_MCTS, simulations=2), max_plies=60, seed=0)
    buf = ReplayBuffer(10_000)
    buf.add(*game_examples(games))
    opt = torch.optim.Adam(net.parameters(), lr=FULL_CFG.learning_rate, weight_decay=FULL_CFG.weight_decay)
    update(net, buf, 2, FULL_CFG, opt, np.random.default_rng(0), device)          # warm-up
    t = time.perf_counter()
    update(net, buf, 10, FULL_CFG, opt, np.random.default_rng(0), device)
    sec_per_step = (time.perf_counter() - t) / 10

    print(f"\none move of {FULL_CFG.games_per_round} games at {FULL_MCTS.simulations} simulations: {sec_per_move:.1f} s "
          f"({sec_per_move / FULL_CFG.games_per_round / FULL_MCTS.simulations * 1e6:.0f} us per game-simulation)")
    print(f"one update step (batch {FULL_CFG.batch_size}): {sec_per_step * 1000:.0f} ms")
    STAGE1 = load_configs("configs/alphazero.yaml")[3] or 9.0   # the real cap (the smoke run has none)
    print("\nprojected real round (moves per game is unknown before training; early self-play games run long):")
    for moves in (80, 150, 250):
        sp = sec_per_move * moves   # the batch shrinks as games finish, so this is an upper bound
        k = update_steps(FULL_CFG.games_per_round * moves, FULL_CFG)
        up = k * sec_per_step
        rounds = STAGE1 * 3600 / (sp + up)
        print(f"  {moves:3d} moves/game: self-play <= {sp / 60:.1f} min + update {k} steps {up / 60:.1f} min "
              f"-> ~{rounds:.0f} rounds ({rounds * FULL_CFG.games_per_round:.0f} games) in {STAGE1:g} GPU-h")
    if torch.cuda.is_available():
        print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    del net, ev, opt
else:
    print("not a smoke test -- skipping the timing probe")

## Training curves
What to look for (the proposed Step 9 gate): finite losses; policy entropy that doesn't crash toward 0 in
the early rounds; value loss falling below ln 2 (0.693, the loss of always guessing 0.5) after the first
rounds -- the value head is learning who wins. More checkmates and fewer games hitting the ply cap are a
sign self-play is getting sharper; self-play results alone can't show strength (it plays itself), the evals can.

In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd

log = pd.read_csv(os.path.join(RUN_DIR, "az_log.csv"))
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
ax[0, 0].stackplot(log["round"], log["white_wins"], log["draws"], log["black_wins"], labels=["White wins", "draws", "Black wins"])
ax[0, 0].set_title("self-play results per round"); ax[0, 0].legend(loc="lower left")
ax[0, 1].plot(log["round"], log["entropy"], "o-"); ax[0, 1].set_title("policy entropy (nats)")
ax[0, 2].plot(log["round"], log["policy_loss"], "o-"); ax[0, 2].set_title("policy loss (vs. search visits)")
ax[1, 0].plot(log["round"], log["value_loss"], "o-")
ax[1, 0].axhline(math.log(2), color="gray", ls="--", label="ln 2 (chance)"); ax[1, 0].legend()
ax[1, 0].set_title("value loss")
ax[1, 1].plot(log["round"], log["avg_plies"], "o-", label="avg plies")
for col, label in [("ends_checkmate", "checkmate"), ("max_plies_draws", "hit the ply cap"),
                   ("ends_repetition", "repetition"), ("ends_insufficient_material", "insufficient material"),
                   ("ends_stalemate", "stalemate")]:
    ax[1, 1].plot(log["round"], log[col], "o-", label=label)
ax[1, 1].legend(fontsize=8); ax[1, 1].set_title("game length and how games ended")
ax[1, 2].plot(log["round"], log["train_gpu_hours"], "o-"); ax[1, 2].set_title("training GPU-hours")
for a in ax.flat:
    a.set_xlabel("round")
plt.tight_layout()
plt.show()
per_round = (log["selfplay_seconds"] + log["update_seconds"]).mean()
print(f"{per_round / 60:.1f} min/round ({log['selfplay_seconds'].mean() / 60:.1f} self-play + "
      f"{log['update_seconds'].mean() / 60:.1f} update)")
print(log.tail(10).to_string(index=False))

## Evaluate checkpoints into `results.csv`
Rate the chosen kept checkpoints against the 2-3 reference members nearest their strength, exactly like
every other arm: `games_per_opponent` games each in batches of `batch_games`, the row saved after every
batch. The agent searches at the training simulation count with no noise and plays the most-visited move;
its search shows up in `sec_per_move` (AlphaZero's inference cost). A rating flagged `OFF SCALE` means it
beat (or lost to) everyone: move `OPPONENTS` up (or down) and rerun. Rated checkpoints are skipped on a rerun.
Skipped in a smoke test.

In [ ]:
from eval.anchors import ResultsTable, build_reference_agents, close_agents
from eval.elo import format_table
from eval.match import evaluate_agent
from train.az_train import gpu_hours_at, load_agent

kept = dict(list_checkpoints(RUN_DIR))
print("kept rounds:", sorted(kept))
ROUNDS = EVAL_ROUNDS if EVAL_ROUNDS is not None else eval_rounds(RUN_DIR, CFG.checkpoint_every)

if SMOKE_TEST:
    print("smoke test -- skipping evaluation")
else:
    eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
    results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
    FIXED = results.fixed()
    print("fixed ratings:", {n: round(v) for n, v in sorted(FIXED.items(), key=lambda kv: kv[1])})
    print("evaluating rounds:", ROUNDS)
    GAMES, BATCH = eval_cfg["games_per_opponent"], eval_cfg["batch_games"]
    opponents = build_reference_agents(results, OPPONENTS, ROOT)
    try:
        for rnd in ROUNDS:
            name = f"{RUN}@r{rnd}"
            if results.role(name) == "model" and int(results.rows[name]["games"]) >= GAMES * len(OPPONENTS):
                print(f"{name}: already evaluated ({results.rows[name]['games']} games) -- skipped")
                continue
            agent = load_agent(kept[rnd], MCTS, name)

            def save(est, recs, rnd=rnd):
                results.log_model(est, recs, run=f"alphazero/{RUN}", step=rnd, seed=CFG.seed,
                                  train_gpu_hours=gpu_hours_at(RUN_DIR, rnd), label_cpu_hours=0.0, gpu=GPU,
                                  note=f"pure self-play AlphaZero, MCTS {MCTS.simulations} sims/move")
                results.save()
                print(format_table([est]))

            evaluate_agent(agent, opponents, FIXED, GAMES, batch_games=BATCH, on_batch=save)
            del agent
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)
    display(pd.read_csv(results.path))

## Elo vs. compute
Every run in `results.csv` on one chart: SFT, each LLM RL run, and AlphaZero as its own series from 0
GPU-hours, with 95% CIs and nearby reference ratings as dashed lines. Saved to
`results/elo_vs_compute.png` on the volume (`eval/plots.py`, reads only `results.csv`).

In [ ]:
from eval.plots import plot_elo_vs_compute

RESULTS_FILE = os.path.join(ROOT, yaml.safe_load(open("configs/eval.yaml"))["results_file"])
PLOT_FILE = os.path.join(ROOT, "results", "elo_vs_compute.png")
if SMOKE_TEST:
    print("smoke test -- skipping the plot")
else:
    fig = plot_elo_vs_compute(RESULTS_FILE, PLOT_FILE)
    display(fig)
    print("saved:", PLOT_FILE)